# SO-101 in Foxglove Colab Embed (JointStates-only)

This notebook demonstrates a simple Foxglove integration for the SO-101 robot model using only `foxglove.JointStates` messages.

What it does:
- Loads the SO-101 URDF into a 3D panel
- Generates sinusoidal joint motion for six joints
- Displays everything in the embedded Foxglove viewer

No robot hardware and no TF publisher are required for this example.

## 1) Install dependencies

Run this cell in Colab (or any Jupyter environment) to install the notebook integration extras.

In [ ]:
%pip install -q "foxglove-sdk[notebook]"

## 2) Configure the SO-101 URDF layout and motion parameters

The URDF is loaded directly from the public GitHub URL and configured to use `jointStates` control mode on `/joint_states`.

In [ ]:
import json
import math
import time

import foxglove
import foxglove.layouts as fl
from foxglove.channels import JointStatesChannel
from foxglove.messages import JointState, JointStates, Timestamp

URDF_URL = "https://raw.githubusercontent.com/TheRobotStudio/SO-ARM100/refs/heads/main/Simulation/SO101/so101_new_calib.urdf"
JOINT_STATES_TOPIC = "/joint_states"

# Joint limits from the SO-101 URDF (radians for revolute joints).
JOINT_LIMITS = {
    "shoulder_pan": (-1.91986, 1.91986),
    "shoulder_lift": (-1.74533, 1.74533),
    "elbow_flex": (-1.69, 1.69),
    "wrist_flex": (-1.65806, 1.65806),
    "wrist_roll": (-2.74385, 2.84121),
    "gripper": (-0.174533, 1.74533),
}

JOINT_PROFILES = {
    "shoulder_pan": {"frequency_hz": 0.10, "phase_rad": 0.0},
    "shoulder_lift": {"frequency_hz": 0.13, "phase_rad": 0.7},
    "elbow_flex": {"frequency_hz": 0.16, "phase_rad": 1.2},
    "wrist_flex": {"frequency_hz": 0.22, "phase_rad": 2.0},
    "wrist_roll": {"frequency_hz": 0.27, "phase_rad": 2.7},
    "gripper": {"frequency_hz": 0.18, "phase_rad": 1.8},
}

layout = fl.Layout(
    content=fl.ThreeDeePanel(
        title="SO-101 JointStates Demo",
        config=fl.ThreeDeeConfig(
            follow_tf="base_link",
            follow_mode="follow-none",
            layers={
                "grid": fl.BaseRendererGridLayerSettings(
                    label="Grid",
                    size=10,
                    divisions=10,
                    line_width=1,
                    color="#248eff",
                    position=(0.0, 0.0, 0.0),
                    rotation=(0.0, 0.0, 0.0),
                    visible=True,
                ),
                "so101_urdf": fl.BaseRendererUrdfLayerSettings(
                    label="SO-101 URDF",
                    source_type="url",
                    url=URDF_URL,
                    control_mode="jointStates",
                    joint_states_topic=JOINT_STATES_TOPIC,
                    display_mode="auto",
                    show_outlines=True,
                    visible=True,
                ),
            },
        ),
    )
)

# Optional helper: exported layout JSON (same config used below).
layout_json = json.loads(layout.to_json())
layout_json

## 3) Generate sinusoidal `JointStates` messages

The next cell buffers about 12 seconds of synthetic SO-101 joint motion at 30 Hz.
Only `foxglove.JointStates` is published (no `/tf` topic).

In [ ]:
RATE_HZ = 30.0
DURATION_SEC = 12.0


def _joint_position_and_velocity(name: str, t_sec: float) -> tuple[float, float]:
    lower, upper = JOINT_LIMITS[name]
    profile = JOINT_PROFILES[name]

    center = 0.5 * (lower + upper)
    amplitude = 0.25 * (upper - lower)

    omega = 2.0 * math.pi * profile["frequency_hz"]
    phase = profile["phase_rad"]

    position = center + amplitude * math.sin(omega * t_sec + phase)
    velocity = amplitude * omega * math.cos(omega * t_sec + phase)
    return position, velocity


foxglove.set_log_level("INFO")
nb_buffer = foxglove.init_notebook_buffer()
joint_channel = JointStatesChannel(topic=JOINT_STATES_TOPIC)

start_time = time.time()
num_samples = int(RATE_HZ * DURATION_SEC)

for sample_idx in range(num_samples):
    t_sec = sample_idx / RATE_HZ
    timestamp = Timestamp.from_epoch_secs(start_time + t_sec)

    joints = []
    for joint_name in JOINT_LIMITS:
        position, velocity = _joint_position_and_velocity(joint_name, t_sec)
        joints.append(
            JointState(
                name=joint_name,
                position=position,
                velocity=velocity,
            )
        )

    joint_channel.log(JointStates(timestamp=timestamp, joints=joints))

print(f"Buffered {num_samples} JointStates messages on {JOINT_STATES_TOPIC}.")

## 4) Show the embedded Foxglove viewer

The viewer below should load the SO-101 URDF and animate the arm using the buffered joint states.

In [ ]:
widget = nb_buffer.show(
    height=640,
    src="https://embed.foxglove.dev/",
    opaque_layout=layout_json,
)
widget

## 5) Next steps for users

- Tune amplitudes/frequencies in `JOINT_PROFILES`.
- Replace synthetic motion with live robot `JointStates` publishing.
- Swap `URDF_URL` to visualize a different robot model.